# Jupyter и научный Python: NumPy и Matplotlib

## Мотивация

Почти вся исследовательская работа в ML начинается в **ноутбуке**: там удобно по шагам проверять идеи и сразу видеть числа и графики рядом с кодом. Тяжёлые вычисления при этом доверяют не циклам Python, а векторизованному **NumPy**. Итог почти всегда нужно показать глазами — это делает **Matplotlib**.

Сегодня соберём этот ежедневный набор: интерактивный ноутбук и его магии → быстрые массивы → графики. На нём держатся все дальнейшие семинары.

> Код сегодня выполняется в **уже готовом окружении** — на учебном сервере или в Colab. Как окружение собирают самому (`python -m venv`, `uv`, файлы зависимостей) — семинар 5.

## 1. Jupyter Notebook: как он устроен

Ноутбук — это последовательность **ячеек**: с кодом или с текстом (Markdown). Код исполняет **ядро** (kernel) — отдельный процесс Python, который хранит состояние между ячейками: переменная, созданная в одной ячейке, видна в следующих. Число `[N]` слева от ячейки — порядковый номер её запуска, а не позиция в ноутбуке.

- `Shift+Enter` — выполнить ячейку и перейти к следующей.
- `Tab` — автодополнение, `Shift+Tab` — подсказка по сигнатуре.
- Kernel → Restart & Run All — перезапустить ядро и выполнить всё сверху вниз.

Значение **последнего выражения** ячейки печатается само, без `print` — этим удобно смотреть промежуточный результат.

### Хорошие практики

В конце работы перезапустите ядро и выполните все ячейки сверху вниз (Kernel → Restart & Run All) — и убедитесь, что ноутбук отрабатывает **без ошибок**. Так проверяется воспроизводимость: итог не должен зависеть от того, в каком порядке вы запускали ячейки во время работы.

### Лучшие практики

Ноутбук удобен для обучения и разведки: запустил маленький кусочек кода — сразу увидел результат. Но когда кода становится много, поддерживать его в ноутбуке практически невозможно — теряются структура, переиспользование и тестируемость. Поэтому «боевой» код принято держать в репозитории: модулями `.py` под контролем версий (а всё чаще — редактируя их кодовыми агентами). Практическое правило: прототип и разведка — в ноутбуке, а устоявшуюся логику переносите в модули репозитория.

In [ ]:
answer = 42                 # переменная останется в памяти ядра
greeting = "hello, kernel"
answer * 2                  # значение последнего выражения печатается без print

#### ❓ **Вопрос**: Почему один и тот же ноутбук может дать разный результат при запуске ячеек в разном порядке?

<details>

<summary><strong>Ответ</strong></summary>

Ядро хранит состояние (значения переменных) и меняет его в том порядке, в котором вы запускаете ячейки, а не в котором они расположены. Если выполнить ячейки не по порядку или несколько раз, значение переменной может разойтись с кодом выше. Надёжная проверка воспроизводимости — Restart & Run All: перезапуск ядра и выполнение всех ячеек сверху вниз.

</details>

## 2. Магии ноутбука: `%` и `%%`

**Магии** — специальные команды IPython-ядра, которых нет в обычном Python:

- **Строчная магия** `%` действует на одну строку: `%timeit`, `%pwd`, `%who`, `%run script.py`.
- **Ячейковая магия** `%%` действует на **всю ячейку** и стоит первой строкой: `%%time`, `%%bash`, `%%writefile file.py`.
- `!команда` выполняет команду shell: `!pip install numpy`, `!nvidia-smi`.

Частые магии: `%timeit` — усреднённый замер времени выражения; `%%time` — время всей ячейки; `%who` / `%whos` — список определённых переменных.

Раньше вывод графиков Matplotlib «включали» магией `%matplotlib inline`. Сейчас она почти везде не нужна — в свежих Jupyter и в Google Colab отрисовка графиков в ноутбук включена по умолчанию.

In [ ]:
!python --version           # ! — команда shell прямо из ноутбука
%who                        # какие переменные уже определены в ядре
%timeit sum(range(100_000)) # усреднённый замер времени выражения

#### ❓ **Вопрос**: Чем строчная магия `%` отличается от ячейковой `%%`?

<details>

<summary><strong>Ответ</strong></summary>

`%` — строчная магия: относится к одной строке (`%timeit выражение`). `%%` — ячейковая: должна стоять первой строкой и относится ко всей ячейке целиком (`%%time`, `%%bash`, `%%writefile`). Поэтому `%timeit` меряет одно выражение, а `%%time` — весь код ячейки.

</details>

### Интерактивные графики: plotly

Ноутбук умеет показывать не только статичные картинки, но и **интерактивный** вывод. Библиотека `plotly` строит графики, которые можно зумить, вращать и наводить курсор на точки прямо в ячейке — в Colab это работает из коробки. Установка: `!pip install plotly`.

In [ ]:
!pip install -q plotly       # в свежем окружении пакета нет
import plotly.express as px

df = px.data.iris()          # встроенный набор данных (таблица pandas)
df.head()

In [ ]:
fig = px.scatter(df, x="sepal_width", y="sepal_length",
                 color="species", size="petal_length",
                 title="Ирисы Фишера — наведи курсор на точку")
fig.show()

## 3. NumPy: массивы и векторизация

`ndarray` — массив чисел одного типа, лежащих в памяти подряд. Ключевые атрибуты — `shape` (форма) и `dtype` (тип элементов); это **атрибуты, а не методы**, пишутся без скобок. Массивы создают через `np.array`, `np.zeros`, `np.ones`, `np.arange`, `np.linspace`.

Операции применяются **поэлементно, без циклов** (векторизация): `a ** 2`, `np.sqrt(a)`, `a + b`. **Broadcasting** «растягивает» массивы совместимых форм: `a + 10` прибавляет скаляр к каждому элементу. Векторизованный код короче и в десятки раз быстрее цикла Python, потому что операция выполняется единым проходом внутри скомпилированного кода над непрерывной памятью.

In [ ]:
import numpy as np

a = np.array([1, 4, 9, 16, 25])
print("shape:", a.shape, "| dtype:", a.dtype)   # атрибуты, без ()
print("linspace:", np.linspace(0, 1, 5))         # 5 точек от 0 до 1 включительно

Операции применяются к массиву целиком, поэлементно — циклы не нужны:

In [ ]:
print("a ** 2  =", a ** 2)      # поэлементно
print("sqrt(a) =", np.sqrt(a))   # np.sqrt — корень (np.square — квадрат!)
print("a + 10  =", a + 10)       # скаляр прибавился к каждому элементу

#### ❓ **Вопрос**: Почему `list(range(5)) + [10]` и `np.arange(5) + 10` дают разное?

<details>

<summary><strong>Ответ</strong></summary>

Для списка Python оператор `+` — это **конкатенация**: `[0,1,2,3,4] + [10]` даёт `[0,1,2,3,4,10]`. Для массива NumPy `+` — **поэлементная** операция, и скаляр `10` по правилам broadcasting прибавляется к каждому элементу: `[10,11,12,13,14]`.

</details>

In [ ]:
big = np.arange(1_000_000)

%timeit sum(i * i for i in range(1_000_000))    # чистый Python — медленно
%timeit (big * big).sum()                        # векторизованный NumPy — быстро

#### ❓ **Вопрос**: Почему `(big * big).sum()` в разы быстрее цикла Python по тем же числам?

<details>

<summary><strong>Ответ</strong></summary>

В цикле Python на каждый элемент создаётся объект и выполняется интерпретируемый код. NumPy хранит числа одного типа подряд в памяти и выполняет операцию единым проходом внутри скомпилированного кода (C), без пооэлементных Python-объектов и накладных расходов интерпретатора.

</details>

### Индексирование и срезы

Как у списков: `x[2]` — элемент, `x[2:5]` — половинчатый интервал (с 2-го по 4-й включительно), `x[-3:]` — три последних. Но у массивов есть то, чего у списков нет — **булева маска**: сравнение даёт массив из `True`/`False`, и им можно индексировать, оставляя только подходящие элементы.

In [ ]:
x = np.arange(10)

print("x[2:5]  =", x[2:5])          # с 2-го по 4-й
print("x[-3:]  =", x[-3:])          # три последних
print("маска   =", x % 2 == 0)      # массив True/False
print("чётные  =", x[x % 2 == 0])   # индексируем маской

### Агрегации по осям

У `sum`, `mean`, `max` есть параметр `axis` — вдоль какой оси «сворачивать». Правило простое: `axis=0` — сворачиваем строки, остаётся результат **по столбцам**; `axis=1` — наоборот. Без `axis` считается по всему массиву сразу.

In [ ]:
M = np.arange(6).reshape(2, 3)
print(M)

print("по столбцам (axis=0):", M.sum(axis=0))
print("по строкам  (axis=1):", M.sum(axis=1))
print("по всей матрице:     ", M.sum())
print("индекс максимума:    ", M.argmax())

#### ❓ **Вопрос**: Почему `M.sum(axis=0)` для матрицы 2×3 даёт три числа, а не два?

<details>

<summary><strong>Ответ</strong></summary>

`axis=0` — это ось строк, и суммирование идёт **вдоль** неё: строки складываются друг с другом и исчезают, а ось столбцов остаётся. Форма `(2, 3)` превращается в `(3,)`. Полезно читать так: «`axis` — это та ось, которая пропадает».

</details>

### Broadcasting: массивы разной формы

Складывать поэлементно можно не только массивы одинаковой формы. **Broadcasting** — правило, по которому NumPy «растягивает» меньший массив до формы большего, не копируя данные в памяти.

Правило одно: формы сравниваются **справа налево**, и по каждой оси размеры должны либо совпадать, либо один из них должен быть равен 1 — такая ось и растягивается.

```
(3, 4)  и  (4,)   -> ок:   (4,) достраивается до (1, 4) и повторяется по строкам
(3, 4)  и  (3,)   -> ошибка: справа 4 против 3
(2, 1)  и  (1, 3) -> ок:   обе растягиваются, результат (2, 3)
```

Это не синтаксический сахар: именно так центрируют данные, нормируют признаки и считают попарные величины — без единого цикла Python.

In [ ]:
table = np.arange(12).reshape(3, 4).astype(float)
col_mean = table.mean(axis=0)            # среднее по столбцам, форма (4,)

print("формы:", table.shape, "и", col_mean.shape)
print(table - col_mean)                  # (3, 4) - (4,) -> вычлось из каждой строки

Ось длины 1 можно создать самому — индексом `None` (синоним `np.newaxis`). Так из двух одномерных массивов получают таблицу всех попарных сумм:

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20])

print((b[:, None] + a[None, :]).shape)   # (2, 1) + (1, 3) -> (2, 3)
print(b[:, None] + a[None, :])

Если формы несовместимы, NumPy не угадывает, а честно падает:

In [ ]:
try:
    np.arange(3) + np.arange(4)          # (3,) и (4,) — справа 3 против 4
except ValueError as error:
    print("ошибка:", error)

#### ❓ **Вопрос**: Почему `table - col_mean` сработало для форм `(3, 4)` и `(4,)`, а `np.arange(3) + np.arange(4)` — нет?

<details>

<summary><strong>Ответ</strong></summary>

Формы сравниваются справа налево. В первом случае: `4` против `4` — совпало, слева у второго массива оси просто нет, она достраивается как `1` и растягивается до `3`. Во втором случае самая правая ось — `3` против `4`: размеры не совпадают и ни один из них не равен 1, растягивать нечего, поэтому `ValueError`.

Практический вывод: когда broadcasting «не работает», сначала печатайте `.shape` обоих массивов — почти всегда ошибка именно в том, что ось оказалась не с той стороны. Развернуть её помогает индекс `None`.

</details>

## 4. Matplotlib: первый график

`matplotlib.pyplot` строит графики. Быстрый способ — вызвать `plt.plot(x, y)`, затем добавить подписи и показать. Для нескольких графиков на фигуре удобен объектный интерфейс: `fig, ax = plt.subplots()`.

Правила читаемого графика: подписать оси (`xlabel`, `ylabel`), дать заголовок (`title`), включить сетку (`grid`), а при нескольких кривых — легенду (`legend`; для неё у `plot` нужен `label=`). Важно: `savefig` вызывают **до** `show()` — `show()` очищает фигуру, и после него файл получится пустым.

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 200)   # много точек → гладкая кривая
plt.plot(x, np.sin(x))
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.show()

In [ ]:
plt.plot(x, np.sin(x), label="sin(x)")     # label нужен легенде
plt.plot(x, np.cos(x), label="cos(x)")
plt.title("sin и cos на [0, 2π]")
plt.legend()
plt.grid(True)
plt.savefig("sin_cos.png", dpi=150)        # СНАЧАЛА сохранить...
plt.show()                                 # ...потом показать

#### ❓ **Вопрос**: Почему `plt.savefig(...)` нужно вызывать до `plt.show()`?

<details>

<summary><strong>Ответ</strong></summary>

`plt.show()` отрисовывает и **очищает** текущую фигуру. Если вызвать `savefig` после `show()`, сохранять будет уже нечего — файл выйдет пустым. Поэтому порядок: сначала `savefig`, затем `show`.

</details>

### Несколько графиков на одной фигуре

Пока графиков один-два, хватает `plt.plot`. Для сетки графиков берут **объектный интерфейс**: `plt.subplots(rows, cols)` возвращает фигуру и массив областей (`axes`), у каждой — свои `plot`, `set_title`, `grid`. `tight_layout()` в конце разносит подписи, чтобы они не наезжали друг на друга.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))

axes[0].plot(x, np.sin(x))
axes[0].set_title("sin(x)")
axes[1].plot(x, np.sin(x) ** 2)
axes[1].set_title("sin²(x)")
fig.tight_layout()
plt.show()

#### ❓ **Вопрос**: Чем `axes[0].set_title("...")` отличается от `plt.title("...")`?

<details>

<summary><strong>Ответ</strong></summary>

`plt.title` действует на «текущую» область — ту, которую matplotlib считает активной. Пока область одна, это удобно, но в сетке из четырёх графиков угадывать текущую опасно. `axes[0]` — прямая ссылка на конкретную область, и метод `set_title` подписывает именно её. Отсюда правило: один график — можно `plt.*`, несколько — только через `axes`.

</details>

### Разные типы графиков и оформление

Matplotlib умеет много типов графиков и тонкую настройку внешнего вида:

- **тип графика**: `plot` (линия), `scatter` (точки), `bar` (столбцы), `hist` (гистограмма), `fill_between` (заливка области);
- **стиль линии** `linestyle`: `"-"` сплошная, `"--"` пунктир (dashed), `":"` точки (dotted), `"-."` штрихпунктир (dash-dot);
- **цвет** `color` (`"crimson"`, `"#1f77b4"`), **толщина** `linewidth`, **маркеры** `marker="o"`;
- **полупрозрачность** `alpha` от 0 до 1 — спасает, когда линии или точки накладываются друг на друга.

In [ ]:
styles = [("-", "сплошная"), ("--", "пунктир"), (":", "точки"), ("-.", "штрихпунктир")]

plt.figure(figsize=(9, 4))
for i, (line_style, name) in enumerate(styles):
    plt.plot(x, np.sin(x - 0.6 * i), linestyle=line_style, linewidth=2, label=name)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
rng = np.random.default_rng(0)
x_points = rng.normal(size=400)
y_points = 0.6 * x_points + rng.normal(size=400) * 0.4

plt.scatter(x_points, y_points, alpha=0.6, edgecolors="k", linewidths=0.3)
plt.xlabel("x")
plt.title("scatter: alpha=0.6 спасает, когда точки накладываются")
plt.show()

### Гистограмма

`plt.hist` разбивает данные на интервалы (`bins`) и показывает, сколько значений попало в каждый. С `density=True` по вертикали не количество, а **плотность**: площадь всех столбиков равна 1 — только в этом виде гистограмму можно сравнивать с теоретической кривой.

In [ ]:
sample = rng.normal(size=10_000)
grid = np.linspace(-4, 4, 200)
density = np.exp(-grid ** 2 / 2) / np.sqrt(2 * np.pi)

plt.hist(sample, bins=50, density=True, alpha=0.6)
plt.plot(grid, density, color="crimson")
plt.title("Гистограмма выборки и теоретическая плотность")
plt.show()

#### ❓ **Вопрос**: Зачем на диаграмме рассеяния полупрозрачность (`alpha`)?

<details>

<summary><strong>Ответ</strong></summary>

Когда точек много и они накладываются, при `alpha=1` всё сливается в сплошное пятно и не видно, где точек больше. Полупрозрачность делает перекрытия темнее, поэтому по насыщенности цвета читается плотность — где данных много, а где мало.

</details>

## 5. Google Colab

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](#)

<!-- TODO: подставить ссылку на ноутбук в Colab в финальной версии — итоговый репозиторий пока не определён. -->

**Google Colab** — это Jupyter в облаке: ноутбук исполняется на удалённой машине Google с бесплатным доступом к GPU/TPU (Runtime → Change runtime type). Основные библиотеки (`numpy`, `matplotlib`, `pandas`, `torch`) уже предустановлены; недостающее ставят через `!pip install ...`. Проверить GPU — `!nvidia-smi`.

**Сессия одноразовая.** Машина выдаётся на несколько часов и отключается при простое, а вместе с ней исчезает всё, что вы в ней сделали: и доустановленные пакеты, и файлы в рабочем каталоге. Отсюда два следствия — строку `!pip install ...` держат прямо в первой ячейке ноутбука, а результаты сохраняют наружу (как именно — вопрос ниже).

Бесплатные GPU при этом не гарантированы и выдаются по остаточному принципу: для долгих расчётов это неудобно, для семинаров и небольших экспериментов — вполне достаточно.

#### ❓ **Вопрос**: Куда сохранять результаты в Colab, чтобы они не пропали после отключения среды?

<details>

<summary><strong>Ответ</strong></summary>

Наружу — рабочий каталог сессии живёт ровно столько же, сколько сама машина. Обычный способ — примонтировать Google Drive и писать в него:

```python
from google.colab import drive
drive.mount('/content/drive')      # файлы появятся в /content/drive/MyDrive
```

Дальше `plt.savefig("/content/drive/MyDrive/plot.png")` — и картинка переживёт закрытие ноутбука. Разовый файл можно просто скачать себе: `files.download("plot.png")` из модуля `google.colab`.

</details>